# VCR — Joint Answer + Rationale Selection (Demo)

A runnable walkthrough of the CLIP + logistic-regression baseline.

This notebook clones the repo, builds features, trains the two classifiers,
evaluates them, and shows a prediction end-to-end. It runs on the small
**dummy dataset** so it works on a free CPU runtime with no downloads.
To run on the real VCR data, see the last section.

## 1. Get the code and install dependencies

In [ ]:
!git clone https://github.com/Utkarzzzz/vcr-project.git
%cd vcr-project
!pip -q install -r requirements.txt

## 2. Create the tiny dummy dataset (stand-in for real VCR)

In [ ]:
!python src/make_dummy_data.py

## 3. See one decoded sample

How a raw VCR record (token lists with object references) becomes plain text.

In [ ]:
import sys; sys.path.append('src')
from vcr_data import load_split

samples = load_split('data/val.jsonl', 'data/vcr1images')
s = samples[0]
print('Question:', s['question'])
for i, a in enumerate(s['answers']):
    mark = '*' if i == s['answer_label'] else ' '
    print(f'  {mark} answer {i}: {a}')

## 4. Build CLIP features, train, and evaluate

In [ ]:
!python src/build_features.py --jsonl data/train.jsonl --out data/train
!python src/build_features.py --jsonl data/val.jsonl   --out data/val
!python src/train.py    --train data/train
!python src/evaluate.py --features data/val

> On the dummy data these numbers are near random (the images carry no real
> signal). On the real VCR set expect roughly 55–65% Q→A.

## 5. Run inference and view a prediction

In [ ]:
!python src/predict.py --jsonl data/test.jsonl --out predictions.json

import json
from PIL import Image
import matplotlib.pyplot as plt

preds = json.load(open('predictions.json'))
test = load_split('data/test.jsonl', 'data/vcr1images')

s, p = test[0], preds[0]
plt.imshow(Image.open(s['image_path'])); plt.axis('off'); plt.show()
print('Q:', s['question'])
print('Predicted answer   :', s['answers'][p['answer']])
print('Predicted rationale:', s['rationales'][p['rationale']])

## 6. Running on the real VCR data

1. Download the dataset from https://visualcommonsense.com/download/ (register first).
2. Upload `train.jsonl`, `val.jsonl`, `test.jsonl` and the `vcr1images/` folder
   into `data/` (or mount Google Drive).
3. Re-run section 4 pointing at the real files. Switch the runtime to a GPU
   (Runtime → Change runtime type → T4) to speed up feature extraction.